# 02 — Validation sweep

Runs sliding-window inference on the validation set for a trained checkpoint,
pickles the raw goal-confidence curves, then sweeps post-processing parameters
(threshold, NMS, smoothing kernel) to reproduce thesis Tables 4.1, 4.2, 4.9–4.11.

**Usage**: set `CHECKPOINT_PATH` and `EXPERIMENT` (which preset produced the checkpoint),
then run top-down.


## 1. Config

In [ ]:
# ============================ EDIT THESE =====================================
CHECKPOINT_PATH = "/home/jinny/aspotting/checkpoints/exp2_run5_aug_no_jitter/<TIMESTAMP>/best.pt"
EXPERIMENT      = "exp2_run5_aug_no_jitter"   # which preset trained this checkpoint
BACKBONE        = "r2plus1d_18"               # r2plus1d_18 | r3d_18 | mc3_18
TASK            = "binary"                    # binary | 4class
# =============================================================================

DATA_DIR    = "/home/jinny/aspotting/dataset"  # SoccerNet root with league/season/game/
OUTPUT_DIR  = f"/home/jinny/aspotting/results/validation_sweeps/{EXPERIMENT}"

# Clip params — must match training (Table 3.3)
FPS, CLIP_SEC, CLIP_FRAMES, CLIP_SIZE = 25, 4.0, 16, (112, 112)
STRIDE_S    = 0.5
BATCH_SIZE  = 64

# Sweep ranges (Table 3.3)
SWEEP_THRESH    = [0.18, 0.20, 0.22, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
SWEEP_NMS_S     = [25.0, 30.0, 35.0, 40.0, 45.0]
SWEEP_SMOOTH_K  = [1, 3, 5, 7, 9]

# Tolerance windows for AP (§3.5.4)
TIGHT_TOLS = [1, 2, 3, 4, 5]
LOOSE_TOLS = list(range(5, 61, 5))
TOL_FIXED  = 5    # for fixed-threshold P/R/F1


## 2. Imports + load model

In [ ]:
import os, json, pickle, itertools
from collections import defaultdict, deque
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models.video import (
    r2plus1d_18, R2Plus1D_18_Weights,
    r3d_18,      R3D_18_Weights,
    mc3_18,      MC3_18_Weights,
)
from tqdm.auto import tqdm

DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN   = np.array([0.43216, 0.394666, 0.37645],  dtype=np.float32)
STD    = np.array([0.22803, 0.22145,  0.216989], dtype=np.float32)

BACKBONES = {"r2plus1d_18": (r2plus1d_18, R2Plus1D_18_Weights.KINETICS400_V1),
             "r3d_18":      (r3d_18,      R3D_18_Weights.KINETICS400_V1),
             "mc3_18":      (mc3_18,      MC3_18_Weights.KINETICS400_V1)}

factory, weights = BACKBONES[BACKBONE]
model = factory(weights=weights)

# Load state dict — handle both {model: ...} and bare state-dict checkpoints,
# and both Linear / Sequential(Dropout, Linear) heads.
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
in_features = model.fc.in_features
if "fc.weight" in state:
    OUT_FEATURES = int(state["fc.weight"].shape[0])
    model.fc = nn.Linear(in_features, OUT_FEATURES)
else:
    OUT_FEATURES = int(state["fc.1.weight"].shape[0])
    model.fc = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(in_features, OUT_FEATURES))
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()

USE_SIGMOID = (OUT_FEATURES == 1)
if TASK == "binary":
    NUM_CLASSES = 2;  CLASS_NAMES = ["background", "goal"]
else:
    NUM_CLASSES = 4;  CLASS_NAMES = ["background", "shots_off", "shots_on", "goal"]

print(f"Loaded   : {CHECKPOINT_PATH}")
print(f"Head     : {'sigmoid 1-out' if USE_SIGMOID else f'softmax {OUT_FEATURES}-out'}")
print(f"Device   : {DEVICE}")


## 2b. Download validation split if missing

In [ ]:
DOWNLOAD_IF_MISSING = True   # set False to skip; SoccerNet downloader skips existing files anyway
SPLITS_TO_FETCH     = ['valid']

if DOWNLOAD_IF_MISSING:
    try:
        from SoccerNet.Downloader import SoccerNetDownloader
    except ImportError:
        raise ImportError("pip install SoccerNet")
    nv_pw = os.environ.get("NV_PASSWORD")
    if not nv_pw:
        raise RuntimeError(
            "Set NV_PASSWORD in env (register at https://www.soccer-net.org/data). "
            "Example: export NV_PASSWORD=...   or set DOWNLOAD_IF_MISSING=False to skip."
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    d = SoccerNetDownloader(LocalDirectory=str(DATA_DIR))
    d.password = nv_pw
    print(f"Downloading SoccerNet splits {SPLITS_TO_FETCH} into {DATA_DIR} …")
    d.downloadGames(files=["Labels-v2.json"],                  split=SPLITS_TO_FETCH)
    d.downloadGames(files=["1_224p.mkv", "2_224p.mkv"],         split=SPLITS_TO_FETCH)
    print("Download step done (existing files were skipped).")


## 3. Helpers — annotation parser, sliding-window inference, post-processing

In [ ]:
SHOT_ON_LABELS  = {"shots on target", "penalty"}
SHOT_OFF_LABELS = {"shots off target"}


def parse_annotations(labels_path):
    """Returns {half: [(seconds, class_id), ...]} for the active TASK."""
    with open(labels_path) as f:
        data = json.load(f)
    events = defaultdict(list)
    for ann in data["annotations"]:
        label = ann.get("label", "").strip().lower()
        half_str, _ = ann["gameTime"].split(" - ")
        half  = int(half_str)
        pos_s = int(ann["position"]) / 1000.0
        if TASK == "binary":
            if label == "goal":
                events[half].append((pos_s, 1))
        else:
            if label == "goal":
                events[half].append((pos_s, 3))
            elif label in SHOT_ON_LABELS:
                events[half].append((pos_s, 2))
            elif label in SHOT_OFF_LABELS:
                events[half].append((pos_s, 1))
    return dict(events)


def _moving_average(arr, k):
    if k <= 1:
        return arr
    pad = k // 2
    padded = np.pad(arr, (pad, pad), mode="edge")
    return np.convolve(padded, np.ones(k, dtype=np.float32) / k, mode="valid")


def sliding_window_inference(video_path):
    """Returns raw_curve = [(center_sec, [class_probs...]), ...] over a half."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    fps_v         = cap.get(cv2.CAP_PROP_FPS) or FPS
    total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step          = max(1, int(CLIP_SEC * fps_v / CLIP_FRAMES))
    window_frames = step * CLIP_FRAMES
    stride_frames = max(1, int(STRIDE_S * fps_v))

    frame_buf, raw = deque(maxlen=window_frames), []
    batch_clips, batch_centers = [], []

    def _flush():
        if not batch_clips: return
        t = torch.stack(batch_clips).float().to(DEVICE)
        with torch.no_grad(), torch.amp.autocast(DEVICE):
            logits = model(t)
            if USE_SIGMOID:
                goal_p = torch.sigmoid(logits[:, 0]).cpu().numpy()
                probs  = np.stack([1 - goal_p, goal_p], axis=1)
            else:
                probs = torch.softmax(logits, dim=1).cpu().numpy()
        for cs, class_probs in zip(batch_centers, probs):
            raw.append((float(cs), class_probs.tolist()))
        batch_clips.clear(); batch_centers.clear()

    first_emit = window_frames - 1
    for frame_idx in tqdm(range(total_frames), desc=Path(video_path).name,
                          leave=False, unit="fr"):
        ok, frame = cap.read()
        if not ok: break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, CLIP_SIZE)
        frame_buf.append(frame)
        if frame_idx >= first_emit and (frame_idx - first_emit) % stride_frames == 0:
            buf = list(frame_buf)
            frames = [buf[i * step] for i in range(CLIP_FRAMES)]
            arr = (np.stack(frames).astype(np.float32) / 255.0 - MEAN) / STD
            clip = torch.from_numpy(arr).permute(3, 0, 1, 2)
            cs = (frame_idx - window_frames // 2) / fps_v
            batch_clips.append(clip); batch_centers.append(cs)
            if len(batch_clips) == BATCH_SIZE:
                _flush()
    cap.release(); _flush()
    return sorted(raw, key=lambda x: x[0])


def postprocess_goal(raw_curve, thresh, nms_s, smooth_k):
    """Post-process the GOAL channel only (used for event-level spotting)."""
    if not raw_curve: return []
    goal_cid = 1 if TASK == "binary" else 3
    cs_arr = [x[0] for x in raw_curve]
    p_arr  = np.array([x[1][goal_cid] for x in raw_curve], np.float32)
    if smooth_k > 1:
        p_arr = _moving_average(p_arr, smooth_k)
    candidates = sorted(zip(cs_arr, p_arr.tolist()), key=lambda x: -x[1])
    dets = []
    for cs, prob in candidates:
        if prob < thresh: continue
        if all(abs(cs - d["timestamp_s"]) >= nms_s for d in dets):
            dets.append({"timestamp_s": cs, "confidence": float(prob)})
    dets.sort(key=lambda x: x["timestamp_s"])
    return dets


## 4. Discover validation games

In [ ]:
valset_root = DATA_DIR
games = []
for labels_path in sorted(valset_root.rglob("Labels-v2.json")):
    game_dir = labels_path.parent
    if any(game_dir.glob("*_224p.mkv")):
        games.append(game_dir)
print(f"Found {len(games)} validation games in {DATA_DIR}")


## 5. Sliding-window inference → pickle raw curves (slow; do once per checkpoint)

In [ ]:
RAW_PICKLE = Path(OUTPUT_DIR) / "raw_curves_val.pkl"
FORCE_RERUN = False

if RAW_PICKLE.exists() and not FORCE_RERUN:
    with open(RAW_PICKLE, "rb") as f:
        saved = pickle.load(f)
    all_raw = saved["all_raw"]
    all_gt  = saved["all_gt"]
    print(f"Loaded cached raw curves from {RAW_PICKLE}  ({len(all_raw)} halves)")
else:
    all_raw, all_gt = {}, {}
    goal_cid = 1 if TASK == "binary" else 3
    for game_dir in tqdm(games, desc="games"):
        annotations = parse_annotations(game_dir / "Labels-v2.json")
        for half in (1, 2):
            vid = game_dir / f"{half}_224p.mkv"
            if not vid.exists():
                continue
            key = (str(game_dir.relative_to(valset_root)), half)
            raw_curve   = sliding_window_inference(str(vid))
            all_raw[key] = raw_curve
            half_evs    = annotations.get(half, [])
            all_gt[key] = [t for t, c in half_evs if c == goal_cid]
    with open(RAW_PICKLE, "wb") as f:
        pickle.dump({"all_raw": all_raw, "all_gt": all_gt,
                     "checkpoint": CHECKPOINT_PATH, "task": TASK}, f)
    print(f"\nSaved raw curves: {RAW_PICKLE}  ({len(all_raw)} halves)")


## 6. AP helpers — Tight Avg-AP, Loose Avg-AP (§3.5)

In [ ]:
try:
    _trapz = np.trapezoid
except AttributeError:
    _trapz = np.trapz


def compute_ap_at_tol(preds_with_conf, gt_timestamps, tol):
    """Standard AP: greedy match within tolerance, then PR integral."""
    if not gt_timestamps:
        return float("nan")
    preds = sorted(preds_with_conf, key=lambda x: -x[1])
    matched, tp_list, fp_list = set(), [], []
    for pred_t, _ in preds:
        best, best_d = None, tol
        for i, gt_t in enumerate(gt_timestamps):
            d = abs(pred_t - gt_t)
            if i not in matched and d <= best_d:
                best_d, best = d, i
        if best is not None:
            matched.add(best); tp_list.append(1); fp_list.append(0)
        else:
            tp_list.append(0); fp_list.append(1)
    tp_cum, fp_cum = np.cumsum(tp_list), np.cumsum(fp_list)
    prec = tp_cum / (tp_cum + fp_cum)
    rec  = tp_cum / len(gt_timestamps)
    prec = np.concatenate([[1.0], prec])
    rec  = np.concatenate([[0.0], rec])
    return float(_trapz(prec, rec))


def evaluate_config(all_raw, all_gt, thresh, nms_s, smooth_k):
    """Returns dict with P, R, F1 (at TOL_FIXED), FP/half, Tight Avg-AP, Loose Avg-AP."""
    preds_all, gt_all = [], []
    tp = fp = fn = 0
    n_halves = len(all_raw)
    for key, raw_curve in all_raw.items():
        dets = postprocess_goal(raw_curve, thresh, nms_s, smooth_k)
        gt   = all_gt.get(key, [])
        preds_all.extend((d["timestamp_s"], d["confidence"]) for d in dets)
        gt_all.extend(gt)
        missed = [g for g in gt
                  if not any(abs(g - d["timestamp_s"]) <= TOL_FIXED for d in dets)]
        tp += len(gt) - len(missed)
        fp += sum(1 for d in dets
                  if not any(abs(d["timestamp_s"] - g) <= TOL_FIXED for g in gt))
        fn += len(missed)
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    tight = float(np.nanmean([compute_ap_at_tol(preds_all, gt_all, t) for t in TIGHT_TOLS]))
    loose = float(np.nanmean([compute_ap_at_tol(preds_all, gt_all, t) for t in LOOSE_TOLS]))
    return {
        "thresh": thresh, "nms_s": nms_s, "smooth_k": smooth_k,
        "Goal_P": round(p, 4), "Goal_R": round(r, 4), "Goal_F1": round(f1, 4),
        "FP_per_half": round(fp / max(n_halves, 1), 2),
        "Tight_AvgAP": round(tight, 4), "Loose_AvgAP": round(loose, 4),
        "TP": tp, "FP": fp, "FN": fn,
    }


## 7. Run sweep → save full table + top-5 by F1 and recall (Tables 4.1/4.2/4.9-4.11)

In [ ]:
combos = list(itertools.product(SWEEP_THRESH, SWEEP_NMS_S, SWEEP_SMOOTH_K))
print(f"Sweeping {len(combos)} (thresh, nms, smooth) combinations…")

rows = []
for thresh, nms_s, smooth_k in tqdm(combos, desc="sweep"):
    rows.append(evaluate_config(all_raw, all_gt, thresh, nms_s, smooth_k))

df = pd.DataFrame(rows)

full_csv = Path(OUTPUT_DIR) / "sweep_full.csv"
top5_f1  = Path(OUTPUT_DIR) / "top5_by_f1.csv"
top5_rec = Path(OUTPUT_DIR) / "top5_by_recall.csv"

df.to_csv(full_csv, index=False)
df.sort_values("Goal_F1", ascending=False).head(5).to_csv(top5_f1,  index=False)
df.sort_values(["Goal_R", "FP_per_half"], ascending=[False, True]).head(5).to_csv(top5_rec, index=False)

print(f"\nFull sweep table : {full_csv}")
print(f"Top 5 by F1      : {top5_f1}")
print(f"Top 5 by recall  : {top5_rec}")
print("\n=== TOP 5 BY F1 ===")
print(df.sort_values("Goal_F1", ascending=False).head(5).to_string(index=False))
print("\n=== TOP 5 BY RECALL (ties broken by FP/half) ===")
print(df.sort_values(["Goal_R", "FP_per_half"], ascending=[False, True]).head(5).to_string(index=False))
